In [3]:
import sys
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [4]:
def read_input():
    data = list(map(int, sys.stdin.read().split()))
    n, m = data[0], data[1]
    idx = 2
    d = [[0] * n for _ in range(n)]
    Q = []
    for i in range(n):
        for j in range(n):
            d[i][j] = data[idx]; idx += 1
    for _ in range(m):
        Q.append((data[idx] - 1, data[idx + 1] - 1))
        idx += 2
    return n, m, d, Q

In [ ]:
def solve_ortools(n, m, d, Q):
    manager = pywrapcp.RoutingIndexManager(n, 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return d[from_node][to_node]

    transit_distance_callback = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_distance_callback)

    def step_callback(index):
        return 1

    transit_step_callback = routing.RegisterUnaryTransitCallback(step_callback)

    routing.AddDimension(
        transit_step_callback,
        0,
        10**7,
        True,
        'Step'
    )
    step_dimension = routing.GetDimensionOrDie('Step')

    for (i, j) in Q:
        step_i = step_dimension.CumulVar(manager.NodeToIndex(i))
        step_j = step_dimension.CumulVar(manager.NodeToIndex(j))
        routing.solver().Add(step_i < step_j)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.FromSeconds(5)
    solution = routing.SolveWithParameters(search_parameters)

    status = routing.status()
    if status != 1:
        print(-1)
        return

    if solution:
        print(solution.ObjectiveValue())
        index = routing.Start(0)
        route = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node + 1)
            index = solution.Value(routing.NextVar(index))
        print(*route)